### Configuración

In [1]:
import pandas as pd
from pathlib import Path
import sys
import warnings

In [2]:
sys.path.append("../src")
from entities import EntityResolver
from graphs import HeterogeneousGraphBuilder

In [3]:
# Ignoramos warnings de spaCy para mantener la salida limpia
warnings.filterwarnings("ignore")

In [4]:
BASE_DIR = Path.cwd().parent
CSV_CHATS = BASE_DIR / "data" / "processed" / "chats_completos.csv"
EDGES_CSV_RAW = BASE_DIR / "data" / "processed" / "grafo_aristas.csv"
EDGES_CSV_CLEAN = BASE_DIR / "data" / "processed" / "grafo_aristas_resuelto.csv"

### Orquestación

In [5]:
if not EDGES_CSV_CLEAN.exists():
    print("Iniciando Pipeline de Grafo Heterogéneo...")
    df_chats = pd.read_csv(CSV_CHATS)
    
    # Fase A: Construcción de la Topología Base
    constructor = HeterogeneousGraphBuilder()
    df_grafo_crudo = constructor.build_raw_graph(df_chats)
    df_grafo_crudo.to_csv(EDGES_CSV_RAW, index=False)
    
    # Fase B: Resolución de Entidades (Entity Resolution)
    resolver = EntityResolver()
    df_grafo_limpio = resolver.consolidar_grafo(
        df_grafo=df_grafo_crudo, 
        columnas_entidades=["Origen", "Destino"]
    )
    df_grafo_limpio.to_csv(EDGES_CSV_CLEAN, index=False)
    print("Pipeline completado. Grafo consolidado y guardado.")

else:
    print("Cargando grafo limpio desde caché...")
    df_grafo_limpio = pd.read_csv(EDGES_CSV_CLEAN)

Iniciando Pipeline de Grafo Heterogéneo...
Modelo NLP 'es_core_news_md' cargado correctamente para extracción de entidades.
Analizando dinámicas conversacionales y construyendo aristas...
Pipeline completado. Grafo consolidado y guardado.


In [6]:
# 1. Cargamos el grafo crudo que sacamos en el paso anterior
df_grafo_crudo = pd.read_csv(EDGES_CSV_RAW)

# 2. Instanciamos nuestra clase (podemos inyectarle nuevos alias si descubrimos más)
resolver = EntityResolver()

# 3. Consolidamos la red matemática
df_grafo_limpio = resolver.consolidar_grafo(
    df_grafo=df_grafo_crudo, 
    columnas_entidades=["Origen", "Destino"]
)

### Auditoria

In [7]:
print(f"\nTotal de aristas únicas en la red: {len(df_grafo_limpio)}")

print("\nTop 5 - Relaciones Físicas (COMUNICA_CON):")
display(df_grafo_limpio[df_grafo_limpio['Tipo_Relacion'] == 'COMUNICA_CON'].head(5))

print("\nTop 5 - Relaciones de Influencia (MENCIONA_A):")
display(df_grafo_limpio[df_grafo_limpio['Tipo_Relacion'] == 'MENCIONA_A'].head(5))


Total de aristas únicas en la red: 519

Top 5 - Relaciones Físicas (COMUNICA_CON):


,Origen,Destino,Tipo_Relacion,Peso
0,MIGUEL PALOMERO,RODOLFO REYES,COMUNICA_CON,94
1,RODOLFO REYES,JULIO MARTINEZ SOLA,COMUNICA_CON,86
2,ROBERTO ROSELLI,RODOLFO REYES,COMUNICA_CON,77
3,JULIO MARTINEZ SOLA,RODOLFO REYES,COMUNICA_CON,76
4,RODOLFO REYES,MIGUEL PALOMERO,COMUNICA_CON,68



Top 5 - Relaciones de Influencia (MENCIONA_A):


,Origen,Destino,Tipo_Relacion,Peso
6,RODOLFO REYES,JULIO MARTINEZ SOLA,MENCIONA_A,30
7,JULIO MARTINEZ SOLA,RODOLFO REYES,MENCIONA_A,29
8,ROBERTO ROSELLI,JULIO MARTINEZ MARTINEZ,MENCIONA_A,23
9,ROBERTO ROSELLI,JULIO MARTINEZ SOLA,MENCIONA_A,22
10,ROBERTO ROSELLI,RODOLFO REYES,MENCIONA_A,22
